# Processing the data (PyTorch)

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [ ]:
%%capture
!pip install datasets evaluate transformers


In [4]:
# Import PyTorch library for tensor operations and neural network computations
import torch
# Import AdamW optimizer - an improved version of Adam optimizer with weight decay regularization
from torch.optim import AdamW
# Import Hugging Face transformers: AutoTokenizer for text preprocessing and AutoModelForSequenceClassification for classification tasks
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Same as before
# Define the pre-trained model checkpoint name (BERT base model, uncased version)
checkpoint = "bert-base-uncased"
# Load the tokenizer associated with the checkpoint - converts text to token IDs and handles special tokens
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
# Load the pre-trained BERT model configured for sequence classification tasks (binary/multi-class classification)
# num_labels=2 specifies binary classification (0 and 1) - this prevents warnings about uninitialized classifier weights
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
# Define a list of two sample sentences to be processed
sequences = [
    "I've been waiting for a HuggingFace course my whole life.",
    "This course is amazing!",
]
# Tokenize the sequences: padding=True ensures all sequences have the same length (pads shorter sequences),
# truncation=True cuts sequences longer than the model's max length, return_tensors="pt" returns PyTorch tensors
batch = tokenizer(sequences, padding=True, truncation=True, return_tensors="pt")

# This is new
# Add ground truth labels to the batch - labels are needed for training (1, 1 means both are positive class)
# In a real scenario, these would be actual labels from your training data
batch["labels"] = torch.tensor([1, 1])

# Initialize the AdamW optimizer with the model's parameters - this will update model weights during training
optimizer = AdamW(model.parameters())
# Zero out gradients from previous iterations - this is crucial to prevent gradient accumulation
# Should be called at the start of each training step
optimizer.zero_grad()
# Forward pass: feed the batch (tokenized inputs + labels) to the model and compute the loss
# The **batch unpacks the dictionary (input_ids, attention_mask, labels, etc.) as keyword arguments
loss = model(**batch).loss
# Backward pass: compute gradients of the loss with respect to all model parameters using automatic differentiation
loss.backward()
# Update step: adjust model weights using the computed gradients according to the optimizer's algorithm
optimizer.step()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
# Test the model using its new, updated weights on a new batch of sentences
model.eval()
test_sequences = [
    "I love learning about transformers models.",
    "The weather is terrible today."
]
test_batch = tokenizer(test_sequences, padding=True, truncation=True, return_tensors="pt")
with torch.no_grad():
    outputs = model(**test_batch)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=1)
    print("Test sequences:", test_sequences)
    print("Model predictions with updated weights:", predictions.tolist())



Test sequences: ['I love learning about transformers models.', 'The weather is terrible today.']
Model predictions with updated weights: [1, 1]


In [6]:
from datasets import load_dataset

raw_datasets = load_dataset("glue", "mrpc")
raw_datasets

README.md: 0.00B [00:00, ?B/s]

mrpc/train-00000-of-00001.parquet:   0%|          | 0.00/649k [00:00<?, ?B/s]

mrpc/validation-00000-of-00001.parquet:   0%|          | 0.00/75.7k [00:00<?, ?B/s]

mrpc/test-00000-of-00001.parquet:   0%|          | 0.00/308k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [7]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0}

In [ ]:
# This line returns the names and types of features (columns)
#  in the training dataset, such as sentence1, sentence2, label, and idx.
raw_train_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

In [12]:
from transformers import AutoTokenizer

checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
# Convert dataset columns to lists - tokenizer expects list[str], not ArrowArray
# Accessing the column directly returns a dataset column object, so we convert it to a list
tokenized_sentences_1 = tokenizer(raw_datasets["train"]["sentence1"][:])  # Convert to list with [:]
tokenized_sentences_2 = tokenizer(raw_datasets["train"]["sentence2"][:])  # Convert to list with [:]

In [ ]:
inputs = tokenizer("This is the first sentence.", "This is the second one.")
inputs

In [ ]:
tokenizer.convert_ids_to_tokens(inputs["input_ids"])

In [ ]:
# Convert dataset columns to lists using [:] slice notation - tokenizer requires list[str] format
# This converts the ArrowArray column to a Python list
tokenized_dataset = tokenizer(
    raw_datasets["train"]["sentence1"][:],  # Convert column to list
    raw_datasets["train"]["sentence2"][:],  # Convert column to list
    padding=True,
    truncation=True,
)

In [ ]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

In [ ]:
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
samples = tokenized_datasets["train"][:8]
samples = {k: v for k, v in samples.items() if k not in ["idx", "sentence1", "sentence2"]}
[len(x) for x in samples["input_ids"]]

In [ ]:
batch = data_collator(samples)
{k: v.shape for k, v in batch.items()}